# Child Nutrition Model Pipeline (Zero-Dependency Deployment)

This notebook trains the Decision Tree model, extracts the label encoders to a JSON file, and uses `m2cgen` to export the trained model as pure, native Python code. This completely eliminates the need for `.pkl` or `.onnx` files and requires zero ML dependencies in the production environment.

In [1]:
!pip install pandas scikit-learn m2cgen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 15.5 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 27.9 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 15.9 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 7.1 MB/s  0:00:046m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [scikit-learn] [scikit-learn]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.55.0 requires pillow<13,>=7.1.0, which is not installed.
streamlit 1.55.0 requires pyarrow>=7.0, which is not installed.
streamlit 1.55.0 requires toml<2,>=0.10.1, which is not installed.
streamlit 1.55.0 requires pandas<3,>=1.4.0, but you have pandas 3.0.1 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade

In [2]:
import pandas as pd
import json
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
import m2cgen as m2c

# 1. Load the dataset
try:
    df = pd.read_csv('data/child_nutrition_data.csv')
except FileNotFoundError:
    df = pd.read_csv('child_nutrition_data.csv')

print(f"Dataset loaded with {df.shape[0]} rows and {df.shape[1]} columns.")

Dataset loaded with 200 rows and 8 columns.


In [3]:
# 2. Initialize encoders and extract mappings
encoders = {
    'Gender': LabelEncoder(),
    'Has_Regular_Meals': LabelEncoder(),
    'Eats_Fruits_Veggies': LabelEncoder(),
    'Clean_Drinking_Water': LabelEncoder(),
    'Nutrition_Status': LabelEncoder()
}

label_mappings = {}

for col, le in encoders.items():
    df[col] = le.fit_transform(df[col])
    mapping = {str(class_name): int(encoded_val) for encoded_val, class_name in enumerate(le.classes_)}
    label_mappings[col] = mapping

# Save the mappings to a standard JSON file
with open('label_encoders.json', 'w') as f:
    json.dump(label_mappings, f, indent=4)

print("✅ Label encodings saved successfully to 'label_encoders.json'")

✅ Label encodings saved successfully to 'label_encoders.json'


In [4]:
# 3. Prepare Features and Target
X = df.drop('Nutrition_Status', axis=1)
y = df['Nutrition_Status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the Decision Tree
clf = DecisionTreeClassifier(random_state=42, max_depth=5)
clf.fit(X_train, y_train)

accuracy = clf.score(X_test, y_test)
print(f"✅ Model trained successfully! Test Accuracy: {accuracy:.2f}")

✅ Model trained successfully! Test Accuracy: 1.00


In [5]:
# 4. Convert and Export to Native Python Code
code = m2c.export_to_python(clf)

with open('model_logic.py', 'w') as f:
    f.write(code)

print("✅ Model successfully exported to 'model_logic.py'")

✅ Model successfully exported to 'model_logic.py'


In [6]:
# 5. Verify m2cgen Inference (Sanity Check)
from model_logic import score

# Take the first row of our test set as a sample list of floats
sample_input = X_test.iloc[0].values.tolist()

# m2cgen returns an array of class scores. The predicted class is the index with the highest score.
m2cgen_output = score(sample_input)
m2cgen_prediction_index = np.argmax(m2cgen_output)

# Predict using Sklearn directly
sklearn_prediction_index = clf.predict([sample_input])[0]

print(f"Sklearn Prediction Index: {sklearn_prediction_index}")
print(f"m2cgen Prediction Index:  {m2cgen_prediction_index}")
assert m2cgen_prediction_index == sklearn_prediction_index, "Predictions do not match!"
print("✅ Verification passed: Native Python code behaves identically to the Sklearn model.")

Sklearn Prediction Index: 2
m2cgen Prediction Index:  2
✅ Verification passed: Native Python code behaves identically to the Sklearn model.


/home/codespace/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
